In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# Vital sign check

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 2*ω_a#7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.4
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 0.0#10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b = 0.0#1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max= 200.0

re_a_0, im_a_0 = 0.0, 0.0
ϕ_a_0, q_a_0 = sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 1.5, -0.5
ϕ_b_0, q_b_0 = sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)
plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false, linewidth=0.5)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_b, q_CL_b, t_list_CL, ω_d, Z_b, phase=pi*0.09)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false, linewidth=0.5)

# No dissipation

In [ ]:
t_max = 100.0
t_min = 0.0
t_min_save = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2*pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max

re_b_0, im_b_0 = 0.84, -0.2
ϕ_b_0, q_b_0 = sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0
environment = MeanField(κ_a, κ_b)
system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, 0.0, ϕ_b_0, q_b_0)

environment_prepared = prepare(system,environment);

In [ ]:
Im_a_list=range(-7.5,7.5,20)
Re_a_list=range(-15,15,20)
Final_mode_a_re = Dict()
Final_mode_a_im = Dict()
Final_mode_b_re = Dict()
Final_mode_b_im = Dict()

ϵ_0 = 0.46
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
for i in ProgressBar(1:length(Re_a_list))
    re_i = Re_a_list[i]
    ϕ_a_0_i=sqrt(2*Z_a)*re_i
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        q_a_0_j = sqrt(2/Z_a)*im_j

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0_i, q_a_0_j, ϕ_b_0, q_b_0)
        system_prepared = prepare(system)

        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver(u_dot_0, u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

        Re_rotating = @.(ϕ_a * cos(ω_a * t_list) + q_a * Z_a * sin(-ω_a * t_list)) / sqrt(2*Z_a)
        Im_rotating = @.(-ϕ_a/Z_a * sin(-ω_a * t_list) + q_a * cos(ω_a * t_list)) * sqrt(Z_a/2)

        Final_mode_a_re[(re_i,im_j)] = Re_rotating
        Final_mode_a_im[(re_i,im_j)] = Im_rotating

        Re_rotating_b = @.(ϕ_b * cos(ω_b_0 * t_list) + q_b * Z_b * sin(-ω_b_0 * t_list)) / sqrt(2*Z_b)
        Im_rotating_b = @.(-ϕ_b/Z_b * sin(-ω_b_0 * t_list) + q_b * cos(ω_b_0 * t_list)) * sqrt(Z_b/2)

        Final_mode_b_re[(re_i,im_j)] = Re_rotating_b
        Final_mode_b_im[(re_i,im_j)] = Im_rotating_b
    end
end



In [ ]:
p0=plot()
c=0
for (re_i, im_j) in keys(Final_mode_a_re)
    c+=1
    scatter!(p0, Final_mode_a_re[(re_i, im_j)], Final_mode_a_im[(re_i, im_j)], label="", color=:black,
    ms=.3)
end
plot!(p0, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz")
display(plot(p0,
xlims=(-20,20),
ylims=(-20,20),top_margin=5mm))


# Test with Harmonic osicllator

In [ ]:
const ∞ = Inf
ħ=1
e=1

ϕ_quantum = 1/2

#### Caracterization of the circuit #####

ω_a = 3.8*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

Z_a=C_a_inv/ω_a # Impedance of the resonator a

κ_a = 0.0

In [ ]:
t_max= 5.0
t_min = 0.0
t_min_save = 0.0


dt_save=1/(5ω_a)
t_save = max(t_min_save,0.0):dt_save:t_max

re_a_0, im_a_0 = 1, 1
ϕ_a_0, q_a_0 = sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0

system = SystemHO(ω_a, E_Ca, ϕ_a_0, q_a_0)
drive = DriveTwoModes(0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
environment = MeanField_HO(κ_a)

ϕ_CL_a, q_CL_a, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false, linewidth=0.5)

In [ ]:
Im_a_list=[-1.5, 0, .5, 1, 1.5]
Re_a_list=[-2.22, -1.5, 0, .5, 1, 1.5, 2, 2.225, 2.5, 3, 3.5]
Final_mode_a_re = Dict()
Final_mode_a_im = Dict()
environment_prepared =prepare(system,environment)
drive_prepared = prepare(drive)
dt= 1/50ω_a
for i in ProgressBar(1:length(Re_a_list))
    re_i = Re_a_list[i]
    ϕ_a_0_i=sqrt(2*Z_a)*re_i
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        q_a_0_j = sqrt(2/Z_a)*im_j

        system = SystemHO(ω_a, E_Ca, ϕ_a_0_i, q_a_0_j)
        system_prepared = prepare(system)

        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        ϕ_a, q_a, t_list = default_solver(u_dot_0, u_0, (t_min, t_max), p, dt, equation=default_equation!)
        Re = ϕ_a / sqrt(2*Z_a)
        Im = q_a * sqrt(Z_a/2)

        Final_mode_a_re[(re_i,im_j)] = Re
        Final_mode_a_im[(re_i,im_j)] = Im
    end
end



In [ ]:
p0=plot()
c=0
for (re_i, im_j) in keys(Final_mode_a_re)
    c+=1
    plot!(p0, Final_mode_a_re[(re_i, im_j)], Final_mode_a_im[(re_i, im_j)], label="", color=:black,
    ms=1, linewidth=0.2)
end
plot!(p0, xlabel=L"\Re(a)", ylabel=L"\Im(a)")
display(plot(p0,
xlims=(-2.5,5),
ylims=(-2.5,2.5),top_margin=5mm))
